In [1]:
import numpy as np
import pandas as pd
from sklearn.svm import SVR
from sklearn.model_selection import train_test_split, KFold, cross_validate, GroupKFold, LeaveOneGroupOut, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score,accuracy_score
import statsmodels.api as sm
from statsmodels.tools.eval_measures import mse, rmse
import unicodedata
import re
import os

In [2]:
# 1. Cargar los datos
BASE_DIR = "."

#ruta_df = os.path.join(BASE_DIR, "df_base_rendimiento_clima_cafe.csv")
ruta_df = os.path.join(BASE_DIR, "Base_data_fe_pca_reduction.csv")

df_total = pd.read_csv(ruta_df)

In [3]:
def logo_seleccion_anio_svm(data, feature_cols, target_col, year_col, min_train_years=10):

    years = sorted(data[year_col].unique())
    results = []
    
    num_cols = ['humedad relativa media anual (%)', 'humedad volumétrica media anual del suelo capa 1 (m³/m³)',
                'spi6_meses_bajo_m1', 'spi12_mean_anual','spi12_meses_bajo_m1','spi3_dic','spi12_dic','spi1_floracion',
                'altitud_media_m','amplitud_temperatura_media_mensual','balance_precipitacion_evaporacion',
                'log1p_área sembrada','log1p_radiación solar acumulada anual (mj/m²/año)','humedad_suelo_diferencia_capa2_capa1',
                'spi3_presion_sequia_anual','spi1_cambio_llenado_vs_floracion','spi3_cambio_llenado_vs_floracion']
    
    # definir parametros para evaluar SVM
    param_grid = {
        'kernel': ['rbf', 'linear'],
        'C': [0.1, 1, 10],
        'epsilon': [0.01, 0.1, 0.5],
        'gamma': ['scale', 'auto']
    }
    
    for i, test_year in enumerate(years):
        if i < min_train_years:
            print(f"Saltamos {test_year}: no hay datos suficientes\n")
            continue
        
        train_years = years[:i]
        
        # =========================
        # 2. Train / test
        # =========================
        train_mask = data[year_col].isin(train_years)
        test_mask = data[year_col] == test_year
        
        X_train = data.loc[train_mask, feature_cols]
        y_train = data.loc[train_mask, target_col]
        X_test = data.loc[test_mask, feature_cols]
        y_test = data.loc[test_mask, target_col]
        
        # =========================
        # 3. Preprocesamiento
        # =========================
        preprocessor = ColumnTransformer(
            transformers=[
                ("num", Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]), num_cols)
            ]
        )
        
        X_train_processed = preprocessor.fit_transform(X_train)
        X_test_processed = preprocessor.transform(X_test)

        # =========================
        # 4. Modelo
        # =========================
        # Grid search para buscar los mejores parametros
        svm_base = SVR()
        grid_search = GridSearchCV(
            svm_base, 
            param_grid, 
            cv=3,
            scoring='neg_mean_absolute_error',
            n_jobs=-1,
            verbose=0
        )
        
        grid_search.fit(X_train_processed, y_train)
        best_svm = grid_search.best_estimator_
        
        # =========================
        # 5. Predicción y evaluación
        # =========================
        predictions = best_svm.predict(X_test_processed)
        
        mae = mean_absolute_error(y_test, predictions)
        r2 = r2_score(y_test, predictions)
        rmse = np.sqrt(mean_squared_error(y_test, predictions))
        
        results.append({
            'test_year': test_year,
            'train_years': train_years,
            'n_train_samples': len(X_train),
            'n_test_samples': len(X_test),
            'mae': mae,
            'r2': r2,
            'rmse': rmse,
            'best_params': grid_search.best_params_,
            'model': best_svm
        })
        
        print(f"   Año {test_year}:")
        print(f"   Años de entrenamiento: {train_years}")
        print(f"   MAE: {mae:.4f}, R²: {r2:.4f}, RMSE: {rmse:.4f}")
        print(f"   Mejores parametros: {grid_search.best_params_}")
    
    return pd.DataFrame(results)



In [4]:
# Run optimized SVM
feature_cols = ['humedad relativa media anual (%)', 'humedad volumétrica media anual del suelo capa 1 (m³/m³)',
                'spi6_meses_bajo_m1', 'spi12_mean_anual','spi12_meses_bajo_m1','spi3_dic','spi12_dic','spi1_floracion',
                'altitud_media_m','amplitud_temperatura_media_mensual','balance_precipitacion_evaporacion',
                'log1p_área sembrada','log1p_radiación solar acumulada anual (mj/m²/año)','humedad_suelo_diferencia_capa2_capa1',
                'spi3_presion_sequia_anual','spi1_cambio_llenado_vs_floracion','spi3_cambio_llenado_vs_floracion']

results_optimized_svm = logo_seleccion_anio_svm(
    df_total,
    feature_cols,
    'rendimiento',
    'anio',
    min_train_years=10
)

print("\n" + "="*50)
print("Resultados:")
print(pd.DataFrame(results_optimized_svm))

Saltamos 2007: no hay datos suficientes

Saltamos 2008: no hay datos suficientes

Saltamos 2009: no hay datos suficientes

Saltamos 2010: no hay datos suficientes

Saltamos 2011: no hay datos suficientes

Saltamos 2012: no hay datos suficientes

Saltamos 2013: no hay datos suficientes

Saltamos 2014: no hay datos suficientes

Saltamos 2015: no hay datos suficientes

Saltamos 2016: no hay datos suficientes

   Año 2017:
   Años de entrenamiento: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016)]
   MAE: 0.2524, R²: -0.5628, RMSE: 0.3468
   Mejores parametros: {'C': 0.1, 'epsilon': 0.01, 'gamma': 'auto', 'kernel': 'rbf'}
   Año 2018:
   Años de entrenamiento: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
   MAE: 0.2338, R²: -0.2918, RMSE: 0.3259
   Me

In [5]:
results_optimized_svm['best_params'][0]

{'C': 0.1, 'epsilon': 0.01, 'gamma': 'auto', 'kernel': 'rbf'}

#### Ajuste Mejor modelo
Con base en el R2 el mejor modelo es el que predice el año 2020 con base en el resto de años (2007 al 2019)

In [6]:
# mejores parametros
results_optimized_svm['best_params'][3]

{'C': 0.1, 'epsilon': 0.01, 'gamma': 'auto', 'kernel': 'rbf'}

In [7]:
years = sorted(df_total['anio'].unique())
results = []
    

feature_cols = ['humedad relativa media anual (%)', 'humedad volumétrica media anual del suelo capa 1 (m³/m³)',
                'spi6_meses_bajo_m1', 'spi12_mean_anual','spi12_meses_bajo_m1','spi3_dic','spi12_dic','spi1_floracion',
                'altitud_media_m','amplitud_temperatura_media_mensual','balance_precipitacion_evaporacion',
                'log1p_área sembrada','log1p_radiación solar acumulada anual (mj/m²/año)','humedad_suelo_diferencia_capa2_capa1',
                'spi3_presion_sequia_anual','spi1_cambio_llenado_vs_floracion','spi3_cambio_llenado_vs_floracion']


target_col='rendimiento'

train_years = years[:-5]
test_year = 2020

# =========================
# 2. Train / test
# =========================
train_mask = df_total['anio'].isin(train_years)
test_mask = df_total['anio'] == test_year
        
X_train = df_total.loc[train_mask, feature_cols]
y_train = df_total.loc[train_mask, target_col]
X_test = df_total.loc[test_mask, feature_cols]
y_test = df_total.loc[test_mask, target_col]
        
# =========================
# 3. Preprocesamiento
# =========================
preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), feature_cols)
    ]
)
        
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# =========================
# 4. Modelo
# =========================
       
model_svm = SVR(kernel='rbf', C=0.1, gamma='auto', epsilon=0.01)
model_svm.fit(X_train_processed, y_train)

# =========================
# 5. Predicción y evaluación
# =========================
y_predict = model_svm.predict(X_test_processed)
        
mae = mean_absolute_error(y_test, y_predict)
r2 = r2_score(y_test, y_predict)
rmse = np.sqrt(mean_squared_error(y_test, y_predict))
        
results.append({
    'test_year': test_year,
    'train_years': train_years,
    'n_train_samples': len(X_train),
    'n_test_samples': len(X_test),
    'mae': mae,
    'r2': r2,
    'rmse': rmse,
})
        
print(f"   Año {test_year}:")
print(f"   Años de entrenamiento: {train_years}")
print(f"   MAE: {mae:.4f}, R²: {r2:.4f}, RMSE: {rmse:.4f}")

   Año 2020:
   Años de entrenamiento: [np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
   MAE: 0.2005, R²: -0.0460, RMSE: 0.2519
